# Week 01 · Day 02 — OpenAI & Anthropic APIs

**Topics:** Authenticated API calls · Streaming · Vision · Tool use · Rate limits


In [ ]:
%pip install -q openai anthropic

In [ ]:
import os, json, time
from openai import OpenAI
import anthropic

oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
ant = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

## 1 · OpenAI Basic Call

In [ ]:
response = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise assistant."},
        {"role": "user", "content": "What are the three laws of thermodynamics? One sentence each."},
    ],
    temperature=0.2,
    max_tokens=200,
)

print(response.choices[0].message.content)
print()
print(f"Tokens used — input: {response.usage.prompt_tokens}, output: {response.usage.completion_tokens}")

## 2 · Anthropic Basic Call

Key differences from OpenAI:
- `max_tokens` is **required**
- System prompt is a separate top-level parameter, not a message
- Response is `message.content[0].text`

In [ ]:
message = ant.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=200,
    system="You are a concise assistant.",
    messages=[
        {"role": "user", "content": "What are the three laws of thermodynamics? One sentence each."},
    ],
)

print(message.content[0].text)
print()
print(f"Tokens used — input: {message.usage.input_tokens}, output: {message.usage.output_tokens}")

## 3 · Streaming

Streaming yields tokens as they're generated. This reduces perceived latency — users see output immediately instead of waiting for the full response.

In [ ]:
# OpenAI streaming
print("OpenAI stream:")
stream = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about machine learning."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

In [ ]:
# Anthropic streaming
print("Anthropic stream:")
with ant.messages.stream(
    model="claude-haiku-4-5-20251001",
    max_tokens=100,
    messages=[{"role": "user", "content": "Write a haiku about machine learning."}],
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
print()

## 4 · Tool Use (Function Calling) — OpenAI

The 4-step cycle:
1. Define tools → send to model
2. Model returns `tool_calls` (no execution)
3. You execute the tool
4. Append assistant message + tool result → send back for final answer

In [ ]:
import json

# Step 1 — define tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["city"],
            },
        },
    }
]

messages = [{"role": "user", "content": "What's the weather in Tokyo in celsius?"}]

# Step 2 — model decides to call the tool
response = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)

assistant_msg = response.choices[0].message
print("Tool call:", assistant_msg.tool_calls[0].function)

In [ ]:
# Step 3 — execute the tool (stub)
def get_weather(city: str, unit: str = "celsius") -> dict:
    return {"city": city, "temperature": 22, "unit": unit, "condition": "sunny"}

tool_call = assistant_msg.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_weather(**args)

# Step 4 — append assistant + tool result, get final answer
messages.append(assistant_msg)  # MUST append assistant message first
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(result),
})

final = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)
print(final.choices[0].message.content)

## 5 · Vision (Multimodal Input)

In [ ]:
# Pass an image URL directly — no download needed
response = oai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in one sentence."},
            {
                "type": "image_url",
                "image_url": {
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
                    "detail": "low",  # low / high / auto — affects cost
                },
            },
        ],
    }],
    max_tokens=100,
)
print(response.choices[0].message.content)

## 6 · Rate Limits and Exponential Backoff

In [ ]:
from openai import RateLimitError

def call_with_retry(messages: list, max_retries: int = 3) -> str:
    for attempt in range(max_retries):
        try:
            r = oai.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                max_tokens=100,
            )
            return r.choices[0].message.content
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = 2 ** attempt  # exponential: 1s, 2s, 4s
            print(f"Rate limited. Retrying in {wait}s...")
            time.sleep(wait)

result = call_with_retry([{"role": "user", "content": "Say hello."}])
print(result)

## Summary

- OpenAI: system prompt as a message; response at `choices[0].message.content`
- Anthropic: system prompt is a top-level param; `max_tokens` required; response at `content[0].text`
- Streaming: reduces perceived latency; yields `delta.content` chunks
- Tool use: 4-step cycle — define → call → execute → return result
- Rate limits: exponential backoff; `2 ** attempt` seconds

**Next:** [Embeddings](week01-day02-embeddings.ipynb)